In [ ]:
import pickle
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
import time

def get_webtoon_urls_by_tab(tab_name, driver):
    """
    특정 요일 또는 '완결' 탭에서 '전체' 버튼을 클릭한 후, 모든 웹툰 URL을 가져오는 함수
    """
    base_url = f"https://webtoon.kakao.com/?tab={tab_name}"
    driver.get(base_url)
    time.sleep(3)  # 페이지 로딩 대기

    print(f"🔍 {tab_name.upper()} 탭 크롤링 시작...")

    # 🔹 "전체" 버튼 클릭 (버튼이 로드될 때까지 기다린 후 클릭)
    try:
        entire_button = WebDriverWait(driver, 10).until(
            EC.element_to_be_clickable((By.XPATH, "//button[span[text()='전체']]"))
        )
        entire_button.click()
        time.sleep(2)  # 클릭 후 페이지 변경 대기
    except Exception as e:
        print(f"❌ '{tab_name.upper()}' 탭에서 '전체' 버튼을 찾을 수 없거나 클릭할 수 없습니다.", e)

    # 🔹 무한 스크롤 (스크롤을 끝까지 내려서 모든 웹툰 로드)
    last_height = driver.execute_script("return document.body.scrollHeight")
    
    while True:
        driver.find_element(By.TAG_NAME, "body").send_keys(Keys.END)  # 페이지 끝으로 스크롤
        time.sleep(2)  # 데이터 로딩 대기
        new_height = driver.execute_script("return document.body.scrollHeight")
        
        if new_height == last_height:  # 더 이상 스크롤이 내려가지 않으면 종료
            break
        last_height = new_height

    # 🔹 웹툰 목록에서 URL 가져오기
    webtoon_urls = []
    webtoon_elements = driver.find_elements(By.CSS_SELECTOR, "a[href*='/content/']")  # 웹툰 링크 요소 찾기
    
    for element in webtoon_elements:
        webtoon_url = element.get_attribute("href")  # 웹툰 URL 가져오기
        if webtoon_url and webtoon_url.startswith("https://webtoon.kakao.com/content/"):
            webtoon_urls.append(webtoon_url)

    print(f"✅ {tab_name.upper()} 탭에서 {len(webtoon_urls)}개의 웹툰을 수집했습니다.\n")
    return list(set(webtoon_urls))  # 중복 제거 후 반환


def get_all_webtoon_urls():
    """
    Selenium을 사용하여 모든 요일 및 완결 탭에서 웹툰 URL을 가져오고 피클 파일로 저장하는 함수
    """
    # 🔹 Chrome WebDriver 설정
    options = webdriver.ChromeOptions()
    options.add_argument("--headless")  # GUI 없이 실행 (백그라운드 모드)
    options.add_argument("--disable-gpu")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")

    driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)

    # 🔹 요일 + 완결 탭 목록
    tabs = ["mon", "tue", "wed", "thu", "fri", "sat", "sun", "complete"]
    all_webtoon_urls = []

    for tab in tabs:
        urls = get_webtoon_urls_by_tab(tab, driver)
        all_webtoon_urls.extend(urls)

    driver.quit()  # 브라우저 종료
    

    all_webtoon_urls = list(set(all_webtoon_urls))  # 중복 제거

    # 🔹 웹툰 URL 리스트를 피클 파일로 저장
    with open("kakao_webtoon_urls.pkl", "wb") as f:
        pickle.dump(all_webtoon_urls, f)

    print(f"✅ 웹툰 URL이 'kakao_webtoon_urls.pkl' 파일로 저장되었습니다.")
    
    return all_webtoon_urls

# ✅ 크롤링 실행 및 저장
webtoon_list = get_all_webtoon_urls()

# ✅ 피클 파일에서 데이터 불러오기 테스트
with open("kakao_webtoon_urls.pkl", "rb") as f:
    loaded_webtoon_list = pickle.load(f)

print(f"📂 저장된 웹툰 URL 개수: {len(loaded_webtoon_list)}개")
for url in loaded_webtoon_list[:10]:  # 상위 10개만 출력
    print(url)


🔍 MON 탭 크롤링 시작...
✅ MON 탭에서 147개의 웹툰을 수집했습니다.

🔍 TUE 탭 크롤링 시작...
✅ TUE 탭에서 150개의 웹툰을 수집했습니다.

🔍 WED 탭 크롤링 시작...
✅ WED 탭에서 143개의 웹툰을 수집했습니다.

🔍 THU 탭 크롤링 시작...
✅ THU 탭에서 149개의 웹툰을 수집했습니다.

🔍 FRI 탭 크롤링 시작...
✅ FRI 탭에서 165개의 웹툰을 수집했습니다.

🔍 SAT 탭 크롤링 시작...
✅ SAT 탭에서 162개의 웹툰을 수집했습니다.

🔍 SUN 탭 크롤링 시작...
✅ SUN 탭에서 127개의 웹툰을 수집했습니다.

🔍 COMPLETE 탭 크롤링 시작...
✅ COMPLETE 탭에서 1657개의 웹툰을 수집했습니다.

✅ 웹툰 URL이 'kakao_webtoon_urls.pkl' 파일로 저장되었습니다.
📂 저장된 웹툰 URL 개수: 2675개
https://webtoon.kakao.com/content/%EB%82%98%EC%81%9C-%EC%8B%9C%EB%85%80%EB%93%A4/4048
https://webtoon.kakao.com/content/%EA%B4%B4%EA%B3%B5%EC%9C%A0%EB%A1%9D/4068
https://webtoon.kakao.com/content/%ED%94%84%EB%A6%B0%EC%84%B8%EC%8A%A4-%EC%BB%A4%EB%84%A5%ED%8A%B8-ReDive/3872
https://webtoon.kakao.com/content/%EC%9A%A9%EC%9D%84-%EA%B7%B8%EB%A6%AC%EB%8A%94-%EC%95%84%EC%9D%B4/4306
https://webtoon.kakao.com/content/%EC%B2%9C%ED%95%98%EC%A0%9C%EC%9D%BC-%EA%B3%A4%EB%A5%9C%EA%B0%9D%EC%9E%94/3977
https://webtoon.kakao.com/content/%EA%B3%B5%ED%8F%A

In [ ]:
import os
import re
import time
import json
from bs4 import BeautifulSoup as bs
from dotenv import load_dotenv
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.action_chains import ActionChains
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

load_dotenv()
# WebDriver 설정
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()))

kakao_id = os.getenv("KAKAO_ID")
kakao_pw = os.getenv("KAKAO_PWD")
driver.get("https://webtoon.kakao.com/more")

# 로그인처리
login_icon = r"#root > main > div > div.bg-background-02 > div > div > a:nth-child(1) > div > img.w-113.h-38"
driver.find_element(By.CSS_SELECTOR, login_icon).click()
time.sleep(2)
login_button = r"body > div:nth-child(5) > div > div > div > div.overflow-x-hidden.overflow-y-auto.\!overflow-hidden.flex.flex-col > div.text-center.pb-\[112px\].overflow-y-auto > div > div > button"
driver.find_element(By.CSS_SELECTOR, login_button).click()
time.sleep(2)

ID_selector = r"#loginId--1"
PWD_selector = r"#password--2"
ID_input = driver.find_element(By.CSS_SELECTOR, ID_selector)
ActionChains(driver).send_keys_to_element(ID_input, kakao_id).perform()
PWD_input = driver.find_element(By.CSS_SELECTOR, PWD_selector)
ActionChains(driver).send_keys_to_element(PWD_input, kakao_pw).perform()
login_selector = (
    r"#mainContent > div > div > form > div.confirm_btn > button.btn_g.highlight.submit"
)
driver.find_element(By.CSS_SELECTOR, login_selector).click()
time.sleep(2)
# 동시접속 기기 횟수 초과 시

alert = r"body > div:nth-child(8) > div > div > div > div.absolute.w-full.px-18.bottom-0.pb-30.flex.left-0.z-1.bg-grey-01.light\:bg-white.Alert_buttonsWrap__2ln9d.pt-10 > button.relative.px-10.py-0.btn-white.light\:btn-black.button-active"
driver.find_element(By.CSS_SELECTOR, alert).click()


In [16]:
import pickle
import requests
import json
import re
import time
from bs4 import BeautifulSoup as bs
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from webdriver_manager.chrome import ChromeDriverManager

def extract_webtoon_id(url):
    """
    카카오웹툰 URL에서 웹툰 ID를 추출하는 함수
    """
    match = re.search(r'/content/.+/(\d+)', url)
    return match.group(1) if match else None

def get_webtoon_price_selenium(webtoon_page_url):
    """
    Selenium을 사용하여 웹툰의 가격 정보를 가져오는 함수
    """
    driver.get(webtoon_page_url)
    time.sleep(2)

    soup = bs(driver.page_source, "html.parser")

    # 🔹 가격 정보가 있는 배지 찾기
    badges = soup.find_all("div", class_="flex flex-wrap gap-4 mb-12")

    for badge in badges:
        texts = badge.find_all("p", class_=lambda x: x and "whitespace-pre-wrap" in x and "font-badge" in x)
        for text in texts:
            text = text.text.strip()

            # 🔹 가격 정보 확인
            if "마다 무료" in text:
                return "기다리면 무료"
            elif "연재" in text:
                return "무료"

    # 🔹 가격 정보가 없으면 기본값 "유료" 반환
    return "유료"


def get_webtoon_info(webtoon_id):
    """
    API를 사용해 웹툰 기본 정보를 가져오고, Selenium을 사용해 추가 정보를 크롤링하는 함수
    """
    profile_url = f"https://gateway-kw.kakao.com/decorator/v2/decorator/contents/{webtoon_id}/profile"
    episode_url = f"https://gateway-kw.kakao.com/episode/v2/views/content-home/contents/{webtoon_id}/episodes?sort=-NO&offset=0&limit=30"

    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/133.0.0.0 Safari/537.36",
        "Referer": "https://webtoon.kakao.com/",
        "Accept-Language": "ko",
    }

    # API 요청
    response = requests.get(profile_url, headers=headers)
    if response.status_code != 200:
        print(f"❌ API 요청 실패: {webtoon_id} (HTTP {response.status_code})")
        return None

    data = response.json().get("data", {})

    # 🔹 웹툰 이름 (seoId) 가져오기
    seo_id = data.get("seoId", "")
    webtoon_page_url = f"https://webtoon.kakao.com/content/{seo_id}/{webtoon_id}?tab=profile"

    # 작가, 일러스트레이터, 원작 분리
    author, illustrator, original = "-", "-", "-"
    for person in data.get("authors", []):
        if person["type"] == "AUTHOR":
            author = person["name"]
        elif person["type"] == "ILLUSTRATOR":
            illustrator = person["name"]
        elif person["type"] == "ORIGINAL_STORY":
            original = person["name"]

    # 연재 상태 설정
    status = "미정"
    for badge in data.get("badges", []):
        if badge["type"] == "STATUS":
            if badge["title"] in ["COMPLETED", "END_OF_SEASON"]:
                status = "완결"
            elif badge["title"] == "EPISODES_PUBLISHING":
                status = "연재"
            elif badge["title"] == "EPISODES_NOT_PUBLISHING":
                status = "휴재"

    # 연재 요일 변환 (기본값 "-")
    update_days = "-"
    weekdays_map = {
        "MON": "월요일", "TUE": "화요일", "WED": "수요일",
        "THU": "목요일", "FRI": "금요일", "SAT": "토요일", "SUN": "일요일"
    }
    days = [weekdays_map.get(b["title"].upper(), "-") for b in data.get("badges", []) if b["type"] == "WEEKDAYS"]
    if days:
        update_days = ", ".join(days)

    # 연령 제한
    age_rating = "전체 이용가"
    for badge in data.get("badges", []):
        if badge["type"] == "AGE_LIMIT":
            age_rating = f"{badge['title']}세 이용가"

    # 키워드 정리
    keywords = ", ".join([k.replace("#", "") for k in data.get("seoKeywords", [])])

    # 가격 정보 설정 (기본값 "-")
    price = "-"
    for badge in data.get("badges", []):
        if badge["type"] == "INFO":
            if badge["title"] == "WAIT_FOR_FREE":
                price = "기다리면 무료"
            elif badge["title"] == "FREE_PUBLISHING":
                price = "무료"

    # 🔹 유료 웹툰인 경우 Selenium으로 가격 크롤링
    if price == "-":
        price = get_webtoon_price_selenium(webtoon_page_url)

    # API에서 에피소드 개수 가져오기
    episode_response = requests.get(episode_url, headers=headers)
    episode_count = episode_response.json().get("meta", {}).get("pagination", {}).get("totalCount", 0) if episode_response.status_code == 200 else 0

    # ✅ Selenium을 이용한 추가 정보 크롤링
    driver.get(webtoon_page_url)
    time.sleep(2)

    soup = bs(driver.page_source, "html.parser")


    # 🔹 장르, 조회수, 별점, 좋아요 가져오기
    genre, views, rating, likes = "-", "-", "-", "-"
    stats = soup.find_all("div", class_="flex justify-center items-start h-14 mt-8 leading-14")
    
    for stat in stats:
        items = stat.find_all("p", class_="whitespace-pre-wrap break-all break-words support-break-word s12-regular-white ml-2 opacity-75")
        if len(items) >= 3:
            genre = items[0].text.strip()  # 장르
            views = items[1].text.strip()  # 조회수
            likes = items[2].text.strip()  # 좋아요

    # 웹툰 정보 통합
    webtoon_info = {
        "id": webtoon_id,
        "type": "웹툰",
        "platform": "카카오웹툰",
        "title": data.get("title", ""),
        "status": status,
        "update_days": update_days,
        "thumbnail": "-",
        "genre": genre,
        "views": views,
        "rating": rating,
        "likes": likes,
        "synopsis": data.get("synopsis", ""),
        "keywords": keywords,
        "author": author,
        "illustrator": illustrator,
        "original": original,
        "age_rating": age_rating,
        "price": price,
        "url": webtoon_page_url,
        "episode_count": episode_count
    }

    return webtoon_info


In [ ]:
import os
import re
import time
import json
import pickle
import requests
from bs4 import BeautifulSoup as bs
from dotenv import load_dotenv
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.action_chains import ActionChains
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager

# 환경 변수 로드 (ID, PW)
load_dotenv()

# Selenium WebDriver 설정 (옵션 제거)

# 🔹 로그인 함수
def kakao_login():
    """
    카카오 웹툰 로그인 수행
    """
    # WebDriver 설정
    driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()))

    kakao_id = os.getenv("KAKAO_ID")
    kakao_pw = os.getenv("KAKAO_PWD")
    driver.get("https://webtoon.kakao.com/more")

    # 로그인처리
    login_icon = r"#root > main > div > div.bg-background-02 > div > div > a:nth-child(1) > div > img.w-113.h-38"
    driver.find_element(By.CSS_SELECTOR, login_icon).click()
    time.sleep(2)
    login_button = r"body > div:nth-child(5) > div > div > div > div.overflow-x-hidden.overflow-y-auto.\!overflow-hidden.flex.flex-col > div.text-center.pb-\[112px\].overflow-y-auto > div > div > button"
    driver.find_element(By.CSS_SELECTOR, login_button).click()
    time.sleep(2)

    ID_selector = r"#loginId--1"
    PWD_selector = r"#password--2"
    ID_input = driver.find_element(By.CSS_SELECTOR, ID_selector)
    ActionChains(driver).send_keys_to_element(ID_input, kakao_id).perform()
    PWD_input = driver.find_element(By.CSS_SELECTOR, PWD_selector)
    ActionChains(driver).send_keys_to_element(PWD_input, kakao_pw).perform()
    login_selector = (
        r"#mainContent > div > div > form > div.confirm_btn > button.btn_g.highlight.submit"
    )
    driver.find_element(By.CSS_SELECTOR, login_selector).click()
    time.sleep(2)
    # 동시접속 기기 횟수 초과 시

    alert = r"body > div:nth-child(8) > div > div > div > div.absolute.w-full.px-18.bottom-0.pb-30.flex.left-0.z-1.bg-grey-01.light\:bg-white.Alert_buttonsWrap__2ln9d.pt-10 > button.relative.px-10.py-0.btn-white.light\:btn-black.button-active"
    driver.find_element(By.CSS_SELECTOR, alert).click()

kakao_login()

# ✅ 2️⃣ 피클 파일에서 웹툰 URL 불러오기
with open("kakao_webtoon_urls.pkl", "rb") as f:
    webtoon_urls = pickle.load(f)

# ✅ 3️⃣ 웹툰 정보 크롤링 실행
webtoon_data_list = []
for url in webtoon_urls[8:10]:  # 5개만 테스트
    webtoon_id = extract_webtoon_id(url)
    if webtoon_id:
        webtoon_data = get_webtoon_info(webtoon_id)
        if webtoon_data:
            webtoon_data_list.append(webtoon_data)

# ✅ 4️⃣ JSON 파일로 저장
with open("kakao_webtoon_data.json", "w", encoding="utf-8") as f:
    json.dump(webtoon_data_list, f, ensure_ascii=False, indent=4)

print(f"✅ 총 {len(webtoon_data_list)}개의 웹툰 정보를 저장했습니다.")


TooManyRedirects: Exceeded 30 redirects.

In [19]:

# ✅ 피클 파일에서 웹툰 URL 불러오기
with open("kakao_webtoon_urls.pkl", "rb") as f:
    webtoon_urls = pickle.load(f)

# ✅ 웹툰 정보 크롤링 실행
webtoon_data_list = []
for url in webtoon_urls[:10]:  # 10개만 테스트
    webtoon_id = extract_webtoon_id(url)
    if webtoon_id:
        webtoon_data = get_webtoon_info(webtoon_id)
        if webtoon_data:
            webtoon_data_list.append(webtoon_data)

# ✅ JSON 파일로 저장
with open("kakao_webtoon_data.json", "w", encoding="utf-8") as f:
    json.dump(webtoon_data_list, f, ensure_ascii=False, indent=4)

print(f"✅ 총 {len(webtoon_data_list)}개의 웹툰 정보를 저장했습니다.")

✅ 총 10개의 웹툰 정보를 저장했습니다.
